<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Statistique descriptive des données
</div>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des données
</div>
Le code ci-dessous permet d'importer ces différents fichiers sous forme de DataFrame Pandas.

In [1]:
import pandas as pd

stations_toulouse = pd.read_parquet("stations_toulouse.parquet") # Données géographiques (Colonnes : 'station', 'longitude', 'latitude', 'station_id')
history_toulouse_train = pd.read_parquet("history_toulouse_train.parquet") # Données temporelles (Index 1 : 'station', Index 2 : 'commit_at', Colonnes : 'bikes' et 'stands')
history_toulouse_test = pd.read_parquet("history_toulouse_test.parquet") # A compléter (Index 1 : 'station', Index 2 : 'commit_at', Colonnes : 'bikes_+15min' à 'bikes_+120min')

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Description des données
</div>
Nombres de stations :

In [2]:
print(len(stations_toulouse))

863


Date de début et fin des mesures par stations :

In [3]:
min_max_date = (history_toulouse_train
            .reset_index(level='commit_at')['commit_at']
            .groupby(level='station')
            .agg(['min', 'max']))

print(min_max_date)

                                                  min                 max
station                                                                  
00001 - POIDS DE L'HUILE          2023-08-05 12:15:00 2026-07-31 21:45:00
00002 - LAFAYETTE                 2023-08-05 12:15:00 2026-07-31 23:45:00
00003 - ALSACE - LORRAINE - POMME 2024-10-31 10:00:00 2025-11-10 11:45:00
00003 - ALSACE-LORRAINE - POMME   2025-11-10 14:30:00 2026-07-31 21:45:00
00003 - POMME                     2023-08-05 12:15:00 2025-01-05 05:45:00
...                                               ...                 ...
NARBONNE - CLOTASSES              2025-08-13 14:15:00 2025-08-22 19:15:00
PASTEUR - LAVOISIER               2025-08-04 10:00:00 2025-08-05 10:15:00
PLACE GEORGES BRASSENS            2025-08-05 09:15:00 2025-08-05 11:00:00
SERVANTY AIRBUS TRAMWAY           2025-07-29 09:15:00 2025-08-01 12:15:00
TEST BEE                          2026-02-10 16:15:00 2026-02-12 16:00:00

[800 rows x 2 columns]


Différences de stations train / test

In [4]:
len(set(history_toulouse_test.index.get_level_values("station")) - set(history_toulouse_train.index.get_level_values("station"))) + len(set(history_toulouse_train.index.get_level_values("station")) - set(history_toulouse_test.index.get_level_values("station")))

0

Stations avec exactement les mêmes coordonnées géographiques (elles ont été renommées)

In [5]:
doublons_coords = stations_toulouse[stations_toulouse.duplicated(subset=["longitude", "latitude"], keep=False)]
print(doublons_coords.sort_values(["longitude", "latitude"]))

                          station  longitude   latitude  station_id
802      1031 - BORNE CGB ATELIER   1.000176  43.615909         802
803   1032 -  BORNE CGB ATELIER 2   1.000176  43.615909         803
805         9002 - HALL COMMINGES   1.331706  43.609075         805
828                HALL COMMINGES   1.331706  43.609075         828
776    03004 - MONTEL - DOUMERGUE   1.342821  43.585029         776
..                            ...        ...        ...         ...
837         MÉRIDIENNE - OCCITANE   1.511495  43.539702         837
779  04002 - LABÈGE INNOPOLE GARE   1.512460  43.547467         779
780  04003 - LABÈGE INNOPOLE GARE   1.512460  43.547467         780
799        07001 - CAYRAS –CATALA   1.517033  43.569408         799
822                CAYRAS –CATALA   1.517033  43.569408         822

[676 rows x 4 columns]


<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Visualisation des données
</div>
Affichage des données sur une carte

In [6]:
import folium

# Centre de la carte = moyenne des coordonnées
center_lat = stations_toulouse["latitude"].mean()
center_lon = stations_toulouse["longitude"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12
)

# Ajouter les stations
for _, row in stations_toulouse.iterrows():
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=f"Station ID : {row['station']}",
        tooltip=f"Station {row['station']}"
    ).add_to(m)

m

si max = 0 : predire 0

clamp entre min station et min station